In [ ]:
import struct
import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"



thincurr_python_path = '/home/clair/repos/install_release'
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.meshing import build_torus_bnorm_grid, ThinCurr_periodic_toroid
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors
from OpenFUSIONToolkit.ThinCurr.valen import VALENSystem 
from OpenFUSIONToolkit.io import histfile

try:
    import pyvista
    pyvista.set_jupyter_backend('static') # Comment to enable interactive PyVista plots
    have_pyvista = True
except ImportError:
    have_pyvista = False

In [ ]:
myOFT = OFT_env(nthreads=2)

tw_torus = ThinCurr(myOFT)
tw_torus.setup_model(mesh_file='thincurr_ex-torus.h5',xml_filename='valen_coils_cyl.xml')
tw_torus.setup_io()


In [ ]:
tw_mode = ThinCurr(myOFT)
tw_mode.setup_model(mesh_file='thincurr_mode.h5', xml_filename='valen_coils_cyl.xml')

with h5py.File('thincurr_mode.h5', 'r+') as h5_file:
    mode_drive = np.asarray(h5_file['thincurr/driver'])

In [ ]:
valen_sys = VALENSystem(tw_torus, tw_mode, mode_drive)

In [ ]:
def run_sa_sweep(valen_sys, s_vals, a_vals, freq=20.0, n=1, coil_phis=None):
    '''! Sweep Boozer s and alpha at a fixed drive frequency and compute the
    coil-driven FR of the plasma-mode block, as in Battey fig. 3

    @param valen_sys VALENSystem instance (already built from tw_torus/tw_mode)
    @param s_vals    Array of Boozer stability parameter values to sweep
    @param a_vals    Array of Boozer torque parameter values to sweep
    @param freq      Drive frequency [Hz] (default 20 Hz, matching fig. 3)
    @param n         Toroidal mode number of the coil drive pattern
    @param coil_phis Toroidal angle of each coil [rad] (default: evenly spaced)
    @result `amp` Response amplitude on the (s,a) grid `[len(s_vals),len(a_vals)]`
    @result `phase` Response phase [deg] on the same grid
    '''
    omega = 2.0*np.pi*freq

    n_wall = valen_sys.Lw.shape[0]
    n_coil = valen_sys.Lc.shape[0]
    n_mode = valen_sys.Ld.shape[0]
    n_total = n_wall + n_coil + n_mode

    # RHS is zero everywhere except the coil block, which carries the mode-n pattern
    coil_driver = np.zeros(n_total,dytpe=complex)
    coil_driver=[np.sin(45*np.pi/180)+1j*np.cos(45*np.pi/180),np.sin(135*np.pi/180)+1j*np.cos(135*np.pi/180),np.sin(225*np.pi/180)+1j*np.cos(225*np.pi/180),np.sin(315*np.pi/180)+1j*np.cos(315*np.pi/180)]

    
    tr = np.zeros((valen_sys.Lwd_ef.shape))
    R_fd = np.array([[valen_sys.Rw,tr],
                     [tr.T, valen_sys.Rd]])

    amp = np.zeros((len(s_vals), len(a_vals)))
    phase = np.zeros((len(s_vals), len(a_vals)))

    for i, s in enumerate(s_vals):
        for j, a in enumerate(a_vals):
            valen_sys.compute_L_ef(s, a)
            L_fd = np.array([[valen_sys.Lw_ef, valen_sys.Lwd_ef],
                             [valen_sys.Ldw_ef, valen_sys.Ld_ef]])
            I = np.linalg.solve(1j*omega*L_fd + R_fd, np.array([valen_sys.Lwc_ef,valen_sys.Ldc_ef]) @ coil_driver)
            Id = I[-2:]
            amp[i, j] = np.linalg.norm(Id)
            phase[i, j] = np.degrees(np.angle(Id[0]))

    return amp, phase


def plot_sa_heatmap(s_vals, a_vals, amp, phase, freq=20.0):
    '''! Plot amplitude and phase heatmaps of the s-alpha sweep, as in fig. 3

    @param s_vals Boozer stability parameter values used in the sweep
    @param a_vals Boozer torque parameter values used in the sweep
    @param amp    Response amplitude grid from `run_sa_sweep`
    @param phase  Response phase grid [deg] from `run_sa_sweep`
    @param freq   Drive frequency [Hz] used in the sweep (for the plot title)
    '''
    S, A = np.meshgrid(s_vals, a_vals, indexing='ij')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

    pcm1 = ax1.pcolormesh(S, A, amp, shading='auto', cmap='viridis')
    ax1.set_xlabel('s (Boozer stability parameter)')
    ax1.set_ylabel(r'$\alpha$ (Boozer torque parameter)')
    ax1.set_title(f'PR amplitude @ {freq:.0f} Hz')
    fig.colorbar(pcm1, ax=ax1)

    pcm2 = ax2.pcolormesh(S, A, phase, shading='auto', cmap='twilight')
    ax2.set_xlabel('s (Boozer stability parameter)')
    ax2.set_ylabel(r'$\alpha$ (Boozer torque parameter)')
    ax2.set_title(f'PR phase @ {freq:.0f} Hz [deg]')
    fig.colorbar(pcm2, ax=ax2)

    plt.tight_layout()
    plt.show()


In [ ]:
s = -0.15
a = 0.05
freq = 20

In [ ]:
omega = 2.0*np.pi*freq

n_wall = valen_sys.Lw.shape[0]
n_coil = valen_sys.Lc.shape[0]
n_mode = valen_sys.Ld.shape[0]
n_total = n_wall + n_coil + n_mode

# RHS is zero everywhere except the coil block, which carries the mode-n pattern
coil_driver = np.zeros(n_total,dytpe=complex)
coil_driver=[np.sin(45*np.pi/180)+1j*np.cos(45*np.pi/180),np.sin(135*np.pi/180)+1j*np.cos(135*np.pi/180),np.sin(225*np.pi/180)+1j*np.cos(225*np.pi/180),np.sin(315*np.pi/180)+1j*np.cos(315*np.pi/180)]


tr = np.zeros((valen_sys.Lwd_ef.shape))
R_fd = np.array([[valen_sys.Rw,tr],
                    [tr.T, valen_sys.Rd]])



valen_sys.compute_L_ef(s, a)
L_fd = np.array([[valen_sys.Lw_ef, valen_sys.Lwd_ef],
                    [valen_sys.Ldw_ef, valen_sys.Ld_ef]])
I = np.linalg.solve(1j*omega*L_fd + R_fd, np.array([valen_sys.Lwc_ef,valen_sys.Ldc_ef]) @ coil_driver)
Id = I[-2:]
amp = np.linalg.norm(Id)
phase = np.degrees(np.angle(Id[0]))
